In [2]:
import pandas as pd
import gensim
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np
from sklearn.model_selection import train_test_split
from nltk.stem import WordNetLemmatizer
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
df = pd.read_csv('spamhamdata.csv',sep='\t', header=None, names=['class', 'text'])
df


,class,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [51]:
from sklearn.preprocessing import LabelEncoder

In [52]:
le = LabelEncoder()
y = le.fit_transform(df['class']) 

In [5]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [6]:
w2v_model = gensim.models.word2vec.Word2Vec.load("Word2Vec-local.bin")

In [7]:
sentences = df['text']
lemmatizer = WordNetLemmatizer()
import re
corpus = []
for i in range(len(sentences)):
    review = re.sub('[^a-zA-Z]', ' ', sentences[i])
    review = review.lower()
    review = review.split()
    review = [lemmatizer.lemmatize(word) for word in review ]
    corpus.append(review)

In [8]:
corpus[1]

['ok', 'lar', 'joking', 'wif', 'u', 'oni']

In [9]:
l=[]
for i in range(len(corpus)):
    l.append(len(corpus[i]))
max(l)

190

In [10]:
vocab_size = len(w2v_model.wv.key_to_index)

In [11]:
vocab_size

3570

In [12]:
max_input_length = max(l)

In [13]:
max_input_length

190

In [14]:
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(df_train['text'])
train_sequences = tokenizer.texts_to_sequences(df_train['text'])
train_padded_sequences = pad_sequences(train_sequences, maxlen=max_input_length, padding='post', truncating='post')

In [15]:
test_sequences = tokenizer.texts_to_sequences(df_test['text'])
test_padded_sequences = pad_sequences(test_sequences, maxlen=max_input_length, padding='post', truncating='post')

In [40]:
y_train = df_train['label_num']

In [33]:
from tensorflow.keras.optimizers import Adam

In [72]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# ... (load your data, Word2Vec model, tokenizer, and hyperparameters like vocab_size, max_input_length) ...

model = Sequential()

# Embedding layer
model.add(Embedding(vocab_size, 100, input_length=max_input_length, 
                    weights=[w2v_model.wv.vectors], trainable=False))

# First LSTM layer
model.add(LSTM(128))  
model.add(Dropout(0.2))  
# Output layer with Softmax 
model.add(Dense(1, activation='sigmoid'))  

# Compile the model
optimizer = Adam(learning_rate=0.1)
model.compile(optimizer=optimizer, 
              loss='binary_crossentropy',  # Use categorical cross-entropy with Softmax
              metrics=['accuracy'])

model.summary()

/Users/sanju/PycharmProjects/NLP_basics/.venv/lib/python3.9/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │       357,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 357,000 (1.36 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 357,000 (1.36 MB)

In [48]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# ... (load your data, Word2Vec model, tokenizer, and hyperparameters like vocab_size, max_input_length) ...

model = Sequential()

# Embedding layer
model.add(Embedding(vocab_size, 100, input_length=max_input_length, 
                    weights=[w2v_model.wv.vectors], trainable=False))

# First LSTM layer
# Second LSTM layer
model.add(LSTM(128))  
model.add(Dropout(0.2))  
# Output layer with Softmax 
model.add(Dense(2, activation='softmax'))  

# Compile the model
optimizer = Adam(learning_rate=0.1)
model.compile(optimizer=optimizer, 
              loss='categorical_crossentropy',  # Use categorical cross-entropy with Softmax
              metrics=['accuracy'])

model.summary()

/Users/sanju/PycharmProjects/NLP_basics/.venv/lib/python3.9/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │       357,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 357,000 (1.36 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 357,000 (1.36 MB)

In [49]:
train_padded_sequences.shape

(4457, 190)

In [50]:
y_train.shape

(4457,)

In [46]:
model.fit(train_padded_sequences,y_train,epochs=3)

Epoch 1/3


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 1), output.shape=(None, 2)

In [47]:
y_test = to_categorical(df_test['label_num'], num_classes=2)

In [26]:
_, accuracy = model.evaluate(test_padded_sequences,y_test)
print('Test Accuracyy: {}'.format(accuracy))

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.8793 - loss: 0.4092
Test Accuracyy: 0.8663676977157593


In [75]:
_, accuracy = model.evaluate(test_padded_sequences,df_test['label_num'])
print('Test Accuracyy: {}'.format(accuracy))

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.8793 - loss: 0.3731
Test Accuracyy: 0.8663676977157593


In [76]:
model.save('Spam_Classifier.keras')

In [29]:
text = "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"
sample_sequences = tokenizer.texts_to_sequences(text)
sample_padded_sequences = pad_sequences(sample_sequences, maxlen=max_input_length, padding='pre', truncating='post')
y_pred = model.predict(sample_padded_sequences)
y_pred = np.argmax(y_pred, axis=1)
print(y_pred)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0]


In [77]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential
from gensim.models import Word2Vec
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load your trained model (replace 'path_to_model' with the actual path)
model = tf.keras.models.load_model('spam_classifier.keras')

# 2. Load the pre-trained Word2Vec model used to create the model (replace 'path_to_w2v_model' with the actual path)
w2v_model = Word2Vec.load('Word2Vec-local.bin') 

# 3. Define the vocabulary size (from the Word2Vec model)
vocab_size = len(w2v_model.wv.key_to_index)

# 4. Create a Tokenizer (using the same vocab_size as the trained model)
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<UNK>') 

# 5. Function to classify a message
def classify_message(message, model, tokenizer, w2v_model, max_length):
    """Classifies a given message as spam or not.

    Args:
        message: The message to classify.
        model: The trained LSTM model.
        tokenizer: The tokenizer used to convert text to sequences.
        w2v_model: The Word2Vec model used for embedding.
        max_length: The maximum input sequence length for the model.

    Returns:
        'spam' or 'ham' based on the model's prediction.
    """
    # Convert message to a sequence
    tokenizer.fit_on_texts([message])
    sequence = tokenizer.texts_to_sequences([message])

    # Pad the sequence to the maximum length
    padded_sequence = pad_sequences(sequence, maxlen=max_input_length, padding='post', truncating='post')

    # Make a prediction using the model
    prediction = model.predict(padded_sequence)[0][0]
    print(prediction)

    # Determine the class label based on the prediction
    if prediction > 0.5:
        return 'spam'
    else:
        return 'ham'
# Get the maximum length from the trained model's Embedding layer
max_length = 190
for i in range(len(df['text'])):
    message = df['text'][i]
    classification = classify_message(message, model, tokenizer, w2v_model, max_input_length)
    if classification == 'spam':
        print(f"The message '{message}' is classified as: {classification}") 
    


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
0.15497516
1/1 ━━━━━━━━━━━━━━━━

KeyboardInterrupt: 

In [ ]:
df['label_num']

In [56]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential
from gensim.models import Word2Vec
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load your trained model (replace 'path_to_model' with the actual path)
model = tf.keras.models.load_model('spam_classifier.keras')

# 2. Load the pre-trained Word2Vec model used to create the model (replace 'path_to_w2v_model' with the actual path)
w2v_model = Word2Vec.load('Word2Vec-local.bin') 

# 3. Define the vocabulary size (from the Word2Vec model)
vocab_size = len(w2v_model.wv.key_to_index)

# 4. Create a Tokenizer (using the same vocab_size as the trained model)
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<UNK>') 

# 5. Function to classify a message
def classify_message(message, model, tokenizer, w2v_model, max_length):
    """Classifies a given message as spam or not.

    Args:
        message: The message to classify.
        model: The trained LSTM model.
        tokenizer: The tokenizer used to convert text to sequences.
        w2v_model: The Word2Vec model used for embedding.
        max_length: The maximum input sequence length for the model.

    Returns:
        'spam' or 'ham' based on the model's prediction.
    """
    # Convert message to a sequence
    tokenizer.fit_on_texts([message])
    sequence = tokenizer.texts_to_sequences([message])
    print(sequence)

    # Pad the sequence to the maximum length
    padded_sequence = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')
    print(padded_sequence)

    # Make a prediction using the model
    prediction = model.predict(padded_sequence)[0]
    print(prediction)

    # Determine the class label based on the prediction (softmax output)
    if prediction[1] > prediction[0]:  # Assuming spam is the second class (index 1)
        return 'spam'
    else:
        return 'ham'

# Get the maximum length from the trained model's Embedding layer
max_length = 190  # Replace with the actual max_length from your model

for i in range(len(df['text'])):
    message = df['text'][i]
    classification = classify_message(message, model, tokenizer, w2v_model, max_length)
    if classification == 'spam':
        print(f"The message '{message}' is classified as: {classification}") 

[[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]]
[[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
[0.88911426 0.11088572]
[[22, 23, 24, 25, 26, 27]]
[[22 23 24 25 26 27  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0

KeyboardInterrupt: 

In [57]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 190, 100)       │       357,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 190, 256)       │       365,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 190, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,076,272 (7.92 MB)

 Trainable params: 573,090 (2.19 MB)

 Non-trainable params: 357,000 (1.36 MB)

 Optimizer params: 1,146,182 (4.37 MB)